***Knowledge extraction***

In [ ]:
import requests
import json

###############################################
# 1. LOAD YOUR CAUSAL EMBEDDINGS
###############################################
# embeddings is numpy array: shape (samples, features)
embeddings =reduced_data_df

###############################################
# 2. SELECT TOP GENES FROM CAUSAL EMBEDDINGS
###############################################
def extract_top_genes(embeddings, top_k=200):
    """
    Select top genes based on variance in the causally-informed embedding.
    High variance → highly expressive / informative in latent causality.
    """
    gene_variances = embeddings.var(axis=0)
    top_genes = gene_variances.sort_values(ascending=False).head(top_k)
    return list(top_genes.index)

top_genes = extract_top_genes(embeddings, top_k=50)
print("[INFO] Number of genes selected for STRING:", len(top_genes))

###############################################
# 3. QUERY STRING DATABASE
###############################################

STRING_API_URL = "https://string-db.org/api"
OUTPUT = "json"
METHOD = "enrichment"
SPECIES = 9606  # human

def string_enrichment(gene_list):
    """
    Query STRING for GO, KEGG, Reactome enrichment.
    """
    params = {
        "identifiers": "%0d".join(gene_list),
        "species": SPECIES
    }

    request_url = f"{STRING_API_URL}/{OUTPUT}/{METHOD}"
    response = requests.post(request_url, data=params)

    if not response.ok:
        raise Exception("STRING API error:", response.text)

    return response.json()

results = string_enrichment(top_genes)

###############################################
# 4. FORMAT THE RESULTS
###############################################
enrichment_df = pd.DataFrame(results)
print("\n=== ENRICHED PATHWAYS FROM STRING ===")
print(enrichment_df[["term", "description", "fdr", "inputGenes"]].head())

###############################################
# 5. SAVE OUTPUT
###############################################
enrichment_df.to_csv("output_file_path", index=True)
print("\n[INFO] Results saved to string_enrichment_results.csv")

In [ ]:
enrichment_df = pd.read_csv("output_file_path", index_col=0)

# Keep top 10 by FDR
top_terms = enrichment_df.sort_values('fdr').head(10)
top_terms['-log10(FDR)'] = -np.log10(top_terms['fdr'])

# Plot
plt.figure(figsize=(8,6))
plt.barh(top_terms['description'], top_terms['-log10(FDR)'], color='skyblue')
plt.xlabel('-log10(FDR)')
plt.ylabel('GO Term / Pathway')
plt.title('Top Enriched Pathways from Causal AE Genes')
plt.gca().invert_yaxis()  # Highest enrichment on top
plt.tight_layout()
plt.show()

In [ ]:
###############################################
# 1. LOAD YOUR CAUSAL EMBEDDINGS
###############################################
# embeddings is numpy array: shape (samples, features)
embeddings1 =reduced_data_df1

top_genes1 = extract_top_genes(embeddings1, top_k=50)
print("[INFO] Number of genes selected for STRING:", len(top_genes1))

###############################################
# 3. QUERY STRING DATABASE
###############################################

results1 = string_enrichment(top_genes1)

###############################################
# 4. FORMAT THE RESULTS
###############################################
enrichment_df1 = pd.DataFrame(results1)
print("\n=== ENRICHED PATHWAYS FROM STRING ===")
print(enrichment_df1[["term", "description", "fdr", "inputGenes"]].head())

###############################################
# 5. SAVE OUTPUT
###############################################
enrichment_df1.to_csv("output_file_path1", index=True)
print("\n[INFO] Results saved to string_enrichment_results1.csv")

In [ ]:
###############################################
# 1. LOAD YOUR CAUSAL EMBEDDINGS
###############################################
# embeddings is numpy array: shape (samples, features)
embeddings2 =reduced_data_df2

###############################################
# 2. SELECT TOP GENES FROM CAUSAL EMBEDDINGS
###############################################

top_genes2 = extract_top_genes(embeddings2, top_k=50)
print("[INFO] Number of genes selected for STRING:", len(top_genes2))

###############################################
# 3. QUERY STRING DATABASE
###############################################

results2 = string_enrichment(top_genes2)

###############################################
# 4. FORMAT THE RESULTS
###############################################
enrichment_df2 = pd.DataFrame(results2)
print("\n=== ENRICHED PATHWAYS FROM STRING ===")
print(enrichment_df2[["term", "description", "fdr", "inputGenes"]].head())

###############################################
# 5. SAVE OUTPUT
###############################################
enrichment_df2.to_csv("output_file_path2", index=True)
print("\n[INFO] Results saved to string_enrichment_results2.csv")

In [ ]:
###############################################
# 1. LOAD YOUR CAUSAL EMBEDDINGS
###############################################
# embeddings is numpy array: shape (samples, features)
embeddings3 =reduced_data_df3

###############################################
# 2. SELECT TOP GENES FROM CAUSAL EMBEDDINGS
###############################################

top_genes3 = extract_top_genes(embeddings3, top_k=50)
print("[INFO] Number of genes selected for STRING:", len(top_genes2))

###############################################
# 3. QUERY STRING DATABASE
###############################################

results3 = string_enrichment(top_genes3)

###############################################
# 4. FORMAT THE RESULTS
###############################################
enrichment_df3 = pd.DataFrame(results3)
print("\n=== ENRICHED PATHWAYS FROM STRING ===")
print(enrichment_df3[["term", "description", "fdr", "inputGenes"]].head())

###############################################
# 5. SAVE OUTPUT
###############################################
enrichment_df3.to_csv("output_file_path3", index=True)
print("\n[INFO] Results saved to string_enrichment_results3.csv")

In [ ]:
# 1) sample sizes and columns for each omics dataframe
for name, df in [("mRNA", df_mRNA), ("Methyl", df_methylation), ("CNA", df_mutation), ("RPPA", df_rppa)]:
    print(name, "shape:", None if df is None else df.shape)
    if df is not None:
        try:
            print(" columns preview:", df.columns[:10])
        except Exception as e:
            print(" columns unavailable:", type(df), e)
    print()

In [ ]:
from collections import Counter

# --- USER: make sure these names match your variables in the notebook ---
# df_mRNA, df_methylation, df_mutation, df_rppa
# payoff_matrix is your {strategy_name: auc} (as shown)
# top_n controls how many top strategies to use
top_n = 10

# Map names (exact strings used in your EGT) -> DataFrame variables
omics_dfs = {
    "mRNA": df_mRNA,
    "Methylation": df_methylation,
    "CNA": df_mutation,       # your variable called df_mutation (CNA)
    "Proteomics": df_rppa
}

# ---------------------------
# Helper functions
# ---------------------------
def clean_and_split_token(token: str):
    """Split multi-gene tokens and clean strings. Returns list of str or []"""
    if pd.isna(token):
        return []
    s = str(token).strip()
    if s.lower() in ("nan", "none", ""):
        return []
    # Split on common separators ; | , / whitespace
    for sep in [';', '|', ',', '/', '\\']:
        if sep in s:
            parts = [p.strip() for p in s.split(sep) if p.strip()]
            # If parts look like 'GENE|ALIAS', prefer the first part that matches a gene-like pattern
            return parts
    # otherwise single token
    return [s]

def df_has_valid_columns(df):
    """Return (is_df, has_non_numeric_cols, sample of cols)"""
    if isinstance(df, pd.DataFrame):
        cols = list(df.columns)
        # detect if columns are gene-like strings or numeric indices
        non_numeric = any(not isinstance(c, (int, np.integer, float)) for c in cols)
        return True, non_numeric, cols[:10]
    else:
        return False, False, None

# ---------------------------
# Diagnostics: check each omics DF
# ---------------------------
print("=== Diagnostics for input DataFrames ===")
for name, df in omics_dfs.items():
    is_df, non_numeric, sample_cols = df_has_valid_columns(df)
    if not is_df:
        print(f"{name}: NOT a pandas DataFrame (type={type(df)}). You must convert to DataFrame with proper columns.")
    else:
        print(f"{name}: shape={df.shape}; columns appear string-like? {non_numeric}; example cols={sample_cols}")

print("\nIf any dataset above is NOT a DataFrame or shows numeric-only columns, fix that first.")
print("Example fix: df = pd.DataFrame(array, columns=original_column_names) or use pd.concat([...], axis=1) when combining.\n")

# ---------------------------
# Recompute top strategies from payoff_matrix (safe)
# ---------------------------
sorted_strategies = sorted(payoff_matrix.items(), key=lambda x: x[1], reverse=True)
top_strategies = [name for name, score in sorted_strategies[:top_n]]
print("Top strategies selected (by AUC):", top_strategies)

# ---------------------------
# Build hub gene list robustly (parse strategy name to determine omics)
# ---------------------------
strategy_to_genes = {}   # map strategy -> list of genes (cleaned)
global_gene_counter = Counter()

for strategy in top_strategies:
    # parse strategy string like "mRNA+CNA+Proteomics"
    parts = strategy.split('+')
    collected = []
    for p in parts:
        p = p.strip()
        if p not in omics_dfs:
            print(f"WARNING: strategy includes '{p}' which is not in omics_dfs mapping.")
            continue
        df = omics_dfs[p]
        # ensure we have a DataFrame
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Omics '{p}' is not a DataFrame (type={type(df)}). Convert so columns contain gene names.")
        # iterate columns and clean tokens
        for col in df.columns:
            for g in clean_and_split_token(col):
                collected.append(g)
    # de-duplicate per-strategy while preserving counts across strategies
    unique_collected = list(dict.fromkeys(collected))
    strategy_to_genes[strategy] = unique_collected
    # increment global counter using unique presence per strategy (so repeated gene in same strategy counts once)
    global_gene_counter.update(unique_collected)

# Diagnostics: print counts
print("\nPer-strategy gene counts (unique):")
for s, genes in strategy_to_genes.items():
    print(f"  {s}: {len(genes)} genes; sample: {genes[:10]}")

print("\nGlobal gene count (unique across top strategies):", len(global_gene_counter))
if len(global_gene_counter) == 0:
    print("\nERROR: No genes collected across top strategies. Likely reasons:")
    print("- The DataFrames used in omics_dfs have numeric column names (not gene symbols).")
    print("- You accidentally converted columns to indices when concatenating (np.concatenate).")
    print("- Column names are 'nan' or empty after cleaning.")
    print("\nFixes:")
    print("1) Ensure each omics df is a pandas DataFrame with gene/probe names as columns.")
    print("2) If you built combined arrays earlier, rebuild them using pd.concat([...], axis=1) to preserve column names.")
    print("3) Show me `df_mRNA.columns[:20]` etc. if you want targeted help.")
    raise SystemExit("Aborting due to empty hub gene list.")

# ---------------------------
# Create final hub gene list and optional weighting
# ---------------------------
# Convert Counter to list ordered by frequency across strategies (descending)
final_hub_genes_by_freq = [g for g, cnt in global_gene_counter.most_common()]
# Choose top_k genes for enrichment (STRING needs >=3; pick top 100 or all)
top_k_for_enrichment = min(200, len(final_hub_genes_by_freq))
final_hub_genes = final_hub_genes_by_freq[:top_k_for_enrichment]

print("\nTop hub genes by frequency across top strategies (sample 50):")
print(final_hub_genes[:50])

# Save or return results as needed
# e.g., write to CSV:
pd.DataFrame(final_hub_genes, columns=["gene"]).to_csv("egt_top_hub_genes", index=False)
print("\nSaved final hub genes to 'egt_top_hub_genes.csv'. Ready to send to STRING enrichment.")


# Overlap test

In [ ]:
from ast import literal_eval
from collections import Counter

def load_enrichment(path):
    """
    Load a STRING-like enrichment CSV.
    Handles inputGenes columns being stringified lists.
    Returns DataFrame with columns: term, description, fdr, inputGenes (Python list)
    """
    df = pd.read_csv(path)
    # normalize column names if needed
    rename_map = {}
    for alt in [('termId','term'), ('name','description'), ('pValue','fdr'), ('genes','inputGenes')]:
        if alt[0] in df.columns and alt[1] not in df.columns:
            rename_map[alt[0]] = alt[1]
    if rename_map:
        df = df.rename(columns=rename_map)
    # ensure inputGenes is list
    if 'inputGenes' in df.columns:
        def parse_cell(x):
            if pd.isna(x): return []
            if isinstance(x, list): return x
            # try literal_eval if looks like list
            try:
                lx = literal_eval(x)
                if isinstance(lx, list): return lx
            except Exception:
                pass
            # fallback: split on common separators
            if isinstance(x, str):
                for sep in [';', ',', '|']:
                    if sep in x:
                        return [s.strip() for s in x.split(sep) if s.strip()]
                return [x.strip()]
            return []
        df['inputGenes'] = df['inputGenes'].apply(parse_cell)
    else:
        df['inputGenes'] = [[]]*len(df)
    return df

In [ ]:
from scipy.stats import hypergeom
import random

def hypergeom_overlap_test(setA, setB, background_size):
    """
    Hypergeometric test for overlap between setA and setB
    background_size = total number of genes universe (e.g., 20000)
    returns observed_overlap, pvalue (prob of >= overlap)
    """
    A = set(setA); B = set(setB)
    k = len(A & B)
    M = background_size
    n = len(A)
    N = len(B)
    # probability of seeing >= k overlaps
    # hypergeom.cdf gives P(X<=k-1), so 1 - cdf(k-1) gives P(X>=k)
    pval = 1 - hypergeom.cdf(k-1, M, n, N)
    return k, pval

def permutation_overlap(setA, setB, background_genes, nperm=1000, seed=0):
    """
    Permutation test: sample random lists of len(B) from background and compute overlap with A.
    Returns empirical p-value and distribution summary.
    """
    rng = random.Random(seed)
    A = set(setA)
    observed = len(A & set(setB))
    counts = []
    bg = list(background_genes)
    nB = len(setB)
    for _ in range(nperm):
        samp = set(rng.sample(bg, nB))
        counts.append(len(A & samp))
    counts = np.array(counts)
    p_emp = (np.sum(counts >= observed) + 1) / (nperm + 1)
    return observed, p_emp, counts

In [ ]:
def pathway_concordance(df_embed, df_egt, top_k=50):
    # pick top_k terms by fdr (lowest)
    t1 = df_embed.sort_values('fdr').head(top_k)
    t2 = df_egt.sort_values('fdr').head(top_k)
    set1 = set(t1['term'].astype(str))
    set2 = set(t2['term'].astype(str))
    jacc = len(set1 & set2) / len(set1 | set2) if (set1 | set2) else 0.0
    return set1, set2, jacc, set1 & set2

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import pearsonr

def pathway_score_matrix(df_features, pathway_genes):
    """
    df_features: samples x features DataFrame (feature names must match pathway_genes)
    pathway_genes: list of genes
    returns vector of pathway score per sample (mean of z-scored features present)
    """
    genes_present = [g for g in pathway_genes if g in df_features.columns]
    if not genes_present:
        return None
    sub = df_features[genes_present]
    # z-score per gene across samples
    subz = (sub - sub.mean()) / (sub.std(ddof=0) + 1e-9)
    score = subz.mean(axis=1)
    return score

def strategy_pathway_relation(df_features, strategy_to_samples, strategy_auc, pathway_genes):
    # compute average pathway score per strategy by averaging sample-level scores
    strat_scores = {}
    for s, samples in strategy_to_samples.items():
        sc = pathway_score_matrix(df_features.loc[samples], pathway_genes)
        if sc is None:
            strat_scores[s] = np.nan
        else:
            strat_scores[s] = np.nanmean(sc)
    # correlate strat_scores (vector) with strategy_auc (vector)
    keys = [s for s in strategy_auc.keys() if s in strat_scores and not np.isnan(strat_scores[s])]
    x = np.array([strat_scores[s] for s in keys])
    y = np.array([strategy_auc[s] for s in keys])
    if len(x) < 3:
        return None
    r, p = pearsonr(x, y)
    return r, p, dict(zip(keys, x))

In [ ]:
import seaborn as sns

def build_pathway_matrix(dfs_by_omics, top_n=30):
    # dfs_by_omics: dict name->enrichment_df
    # pick union of top_n terms per df
    terms = set()
    for k,df in dfs_by_omics.items():
        terms |= set(df.sort_values('fdr').head(top_n)['term'].astype(str).tolist())
    terms = list(terms)
    mat = pd.DataFrame(index=terms, columns=list(dfs_by_omics.keys()), data=np.nan)
    for k,df in dfs_by_omics.items():
        tmp = df.set_index('term')
        for t in terms:
            if t in tmp.index:
                mat.loc[t,k] = -np.log10(tmp.loc[t,'fdr'] + 1e-300)
    # replace NaN with 0
    mat = mat.fillna(0)
    plt.figure(figsize=(8, max(4, len(terms)/6)))
    sns.heatmap(mat, cmap='magma', linewidths=.5)
    plt.title('Pathway significance (-log10 FDR) across embeddings and EGT')
    plt.tight_layout()
    plt.show()
    return mat

In [ ]:
# load files
embed_mrna = load_enrichment("output_file_path")
embed_methyl = load_enrichment("output_file_path1")
embed_cna  = load_enrichment("output_file_path3")
embed_rppa = load_enrichment("output_file_path2")
egt_enr    = load_enrichment("egt_top_hub_genes")

# get top gene lists (pick top_k by fdr)
def top_genes_from_enrich(df, top_k=50):
    # Check if 'fdr' column exists, otherwise sort by 'gene' column or skip sorting
    if 'fdr' in df.columns:
        sorted_df = df.sort_values('fdr').head(top_k)
    else:
        # If no 'fdr' column, assume it's a list of genes and take the first 'top_k'
        # Or, if 'gene' column exists, use that directly.
        if 'gene' in df.columns:
            sorted_df = df.head(top_k) # Assuming the 'gene' column is already sorted or order doesn't matter for picking top_k
        else:
            # Fallback if neither 'fdr' nor 'gene' column is found
            return [] # or handle as appropriate

    all_genes = set()
    for row in sorted_df['inputGenes']:
        for g in row:
            all_genes.add(g)
    return list(all_genes)

# Special handling for egt_enr since it's just a list of genes, not an enrichment result
def get_genes_from_egt_hub(df, top_k=200):
    if 'gene' in df.columns:
        return df['gene'].head(top_k).tolist()
    return []

g_mrna = top_genes_from_enrich(embed_mrna, 50)
g_methyl = top_genes_from_enrich(embed_methyl, 50)
g_cna  = top_genes_from_enrich(embed_cna, 50)
g_rppa = top_genes_from_enrich(embed_rppa, 50)
g_egt  = get_genes_from_egt_hub(egt_enr, 200)  # Use the specialized function for egt_enr

# background universe = union of all measured genes
background = set(g_mrna) | set(g_methyl) | set(g_cna) | set(g_rppa) | set(g_egt)  # or entire assay gene list

# hypergeom test example
k, p = hypergeom_overlap_test(g_mrna, g_egt, background_size=len(background))
print("mRNA vs EGT overlap:", k, "p=", p)

# permutation
obs, p_emp, dist = permutation_overlap(g_mrna, g_egt, background, nperm=2000)
print("perm p_emp=", p_emp)

# pathway concordance heatmap
df_dict = {"mRNA": embed_mrna,  "methylation": embed_methyl, "CNA": embed_cna, "RPPA": embed_rppa}
mat = build_pathway_matrix(df_dict, top_n=40)


In [ ]:
# hypergeom test example
k, p = hypergeom_overlap_test(g_methyl, g_egt, background_size=len(background))
print("Methylation vs EGT overlap:", k, "p=", p)

# permutation
obs, p_emp, dist = permutation_overlap(g_methyl, g_egt, background, nperm=2000)
print("perm p_emp=", p_emp)

In [ ]:
# hypergeom test example
k, p = hypergeom_overlap_test(g_cna, g_egt, background_size=len(background))
print("CNA vs EGT overlap:", k, "p=", p)

# permutation
obs, p_emp, dist = permutation_overlap(g_cna, g_egt, background, nperm=2000)
print("perm p_emp=", p_emp)

In [ ]:
# hypergeom test example
k, p = hypergeom_overlap_test(g_rppa, g_egt, background_size=len(background))
print("RPPA vs EGT overlap:", k, "p=", p)

# permutation
obs, p_emp, dist = permutation_overlap(g_rppa, g_egt, background, nperm=2000)
print("perm p_emp=", p_emp)